# Unit 2 Assignment: Building a Mixture of Experts (MoE) Router

## Smart Customer Support Router

**Objective:** Build a router that classifies user queries and forwards them to specialized expert configurations using the Groq API.

**Experts:**
1. **Technical Expert** — Bug reports, code issues, errors
2. **Billing Expert** — Refund requests, charges, subscriptions
3. **General Expert** — Casual chat, fallback

**Bonus:** Tool Use Expert — Routes data-fetching queries to mock functions

## 1. Setup — Install Dependencies

In [5]:
!pip install -q groq python-dotenv

In [6]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
from getpass import getpass

# Load environment variables from .env file
load_dotenv()

# If GROQ_API_KEY is not set, prompt the user
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("✅ Groq client initialized successfully!")

✅ Groq client initialized successfully!


## 2. Define Expert Configurations

Each "expert" is simulated by using a different **System Prompt** with the same base model (`llama-3.1-8b-instant`). The system prompt shapes the LLM's persona and behavior for each domain.

In [7]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a Senior Technical Support Engineer. You are rigorous, code-focused, and precise. "
            "When a user reports a bug or error, you:\n"
            "1. Identify the root cause of the issue.\n"
            "2. Provide a clear, step-by-step fix with code examples.\n"
            "3. Explain WHY the error occurred so the user learns.\n"
            "Always format code in proper code blocks. Be concise and technical."
        ),
        "temperature": 0.7,
    },
    "billing": {
        "system_prompt": (
            "You are a Senior Billing & Customer Success Specialist. You are empathetic, patient, "
            "and well-versed in company financial policies. When a user has a billing concern, you:\n"
            "1. Acknowledge the inconvenience sincerely.\n"
            "2. Clearly explain the relevant policy (refund window, proration, etc.).\n"
            "3. Provide exact next steps the user should take.\n"
            "Always be warm and professional. Never be dismissive of financial concerns."
        ),
        "temperature": 0.7,
    },
    "general": {
        "system_prompt": (
            "You are a friendly and helpful Customer Support Assistant. You handle general inquiries, "
            "greetings, and questions that don't fall into technical or billing categories. "
            "Be conversational, helpful, and guide the user to the right resource if needed."
        ),
        "temperature": 0.7,
    },
}

print("✅ Expert configurations defined:")
for expert, config in MODEL_CONFIG.items():
    print(f"   - {expert.capitalize()} Expert")

✅ Expert configurations defined:
   - Technical Expert
   - Billing Expert
   - General Expert


## 3. The Router — Intent Classification

The router is the "gating function" of our MoE architecture. It uses an LLM call with `temperature=0` (for deterministic output) to classify the user's query into one of the expert categories.

**Key constraint:** The router must return *only* the category name — nothing else.

In [8]:
VALID_CATEGORIES = list(MODEL_CONFIG.keys())  # ["technical", "billing", "general"]

ROUTER_SYSTEM_PROMPT = (
    "You are an intent classification engine. Your job is to classify user messages "
    "into exactly ONE of these categories: technical, billing, general.\n\n"
    "Rules:\n"
    "- 'technical': Bug reports, code errors, software issues, API problems, debugging.\n"
    "- 'billing': Charges, refunds, subscriptions, invoices, payment issues, pricing.\n"
    "- 'general': Greetings, casual chat, general questions, anything else.\n\n"
    "You MUST respond with ONLY a single lowercase word from: technical, billing, general.\n"
    "Do NOT include any explanation, punctuation, or extra text."
)


def route_prompt(user_input: str) -> str:
    """
    Classify the user's input into a category using an LLM call.
    Returns one of: 'technical', 'billing', 'general'.
    """
    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
        ],
        temperature=0,       # Deterministic — we want consistent routing
        max_tokens=10,       # We only need a single word
    )

    category = response.choices[0].message.content.strip().lower()

    # Fallback to "general" if the model returns something unexpected
    if category not in VALID_CATEGORIES:
        print(f"⚠️  Router returned unexpected category '{category}', falling back to 'general'.")
        category = "general"

    return category


# Quick test of the router
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey, what's up?",
]

print("🔀 Router Test Results:")
print("-" * 60)
for query in test_queries:
    category = route_prompt(query)
    print(f"  Query: \"{query}\"")
    print(f"  → Routed to: [{category.upper()}]\n")

🔀 Router Test Results:
------------------------------------------------------------
  Query: "My python script is throwing an IndexError on line 5."
  → Routed to: [TECHNICAL]

  Query: "I was charged twice for my subscription this month."
  → Routed to: [BILLING]

  Query: "Hey, what's up?"
  → Routed to: [GENERAL]



## 4. The Orchestrator — End-to-End Pipeline

The orchestrator ties everything together:
1. **Routes** the query to identify the category.
2. **Selects** the matching expert's system prompt.
3. **Calls** the LLM with that expert persona.
4. **Returns** the expert's response.

In [9]:
def process_request(user_input: str) -> str:
    """
    Main orchestrator function:
    1. Routes the query to determine category.
    2. Loads the appropriate expert system prompt.
    3. Calls the LLM with the expert configuration.
    4. Returns the expert's response.
    """
    # Step 1: Route the query
    category = route_prompt(user_input)
    expert_config = MODEL_CONFIG[category]

    print(f"🔀 Router Decision: [{category.upper()}]")
    print(f"🤖 Activating {category.capitalize()} Expert...")
    print("=" * 60)

    # Step 2: Call the expert LLM with the appropriate system prompt
    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert_config["temperature"],
        max_tokens=1024,
    )

    answer = response.choices[0].message.content
    return answer

## 5. Testing the Full Pipeline

Let's run several test queries through the complete MoE system and observe how the router directs each query to the appropriate expert.

In [10]:
# Test Case 1: Technical Query
print("=" * 60)
print("📩 TEST 1 — Technical Query")
print("=" * 60)
query1 = "My python script is throwing an IndexError on line 5."
response1 = process_request(query1)
print(response1)

📩 TEST 1 — Technical Query
🔀 Router Decision: [TECHNICAL]
🤖 Activating Technical Expert...
**Error Analysis**

To troubleshoot the `IndexError`, I need more information about your script. However, I'll provide a general outline of steps to identify the issue.

**Code Snippet (Assumption)**

```python
my_list = [1, 2, 3]
index = 5

try:
    value = my_list[index]
except IndexError as e:
    print(f"IndexError occurred: {e}")
```

**Root Cause**

An `IndexError` typically occurs when you try to access an index in a list that is out of its bounds. In Python, list indices start at 0, so if you try to access an index equal to or greater than the length of the list, you'll get an `IndexError`.

**Fix**

To fix the `IndexError`, you need to ensure that the index you're trying to access is within the bounds of the list.

```python
my_list = [1, 2, 3]
index = 2  # changed from 5 to 2

try:
    value = my_list[index]
    print(value)  # print the value at the valid index
except IndexError as e:


In [11]:
# Test Case 2: Billing Query
print("=" * 60)
print("📩 TEST 2 — Billing Query")
print("=" * 60)
query2 = "I was charged twice for my subscription this month."
response2 = process_request(query2)
print(response2)

📩 TEST 2 — Billing Query
🔀 Router Decision: [BILLING]
🤖 Activating Billing Expert...
I'm so sorry to hear that you've been charged twice for your subscription this month. I can imagine how frustrating that must be for you. 

Let's take a look into this right away. Our billing system is designed to prevent duplicate charges, but occasionally errors can occur. I'd like to help resolve this issue for you.

According to our company's financial policy, we offer a 7-day refund window for any duplicate charges. If you'd like to request a refund for the extra charge, please submit a request to our billing department through our support portal within the next 7 days. 

To do this, please follow these steps:

1. Log in to your account at our support portal ([support portal link]).
2. Click on the 'Billing' section.
3. Select 'Request Refund' and fill out the form with your order number and a brief description of the issue.
4. Submit your request.

If you've already submitted a request or have an

In [12]:
# Test Case 3: General Query
print("=" * 60)
print("📩 TEST 3 — General Query")
print("=" * 60)
query3 = "Hey there! What can you help me with?"
response3 = process_request(query3)
print(response3)

📩 TEST 3 — General Query
🔀 Router Decision: [GENERAL]
🤖 Activating General Expert...
Hello! It's great to have you reach out. I'm here to help with any non-technical or billing-related questions you might have. Whether you're looking for information on our services, need assistance with a specific issue, or just want to know more about what we offer, I'm here to provide you with helpful guidance.

What's on your mind? What can I help you with today?


## 6. Bonus Challenge — Tool Use Expert

We add a fourth category: **`tool`** — for queries that require fetching external data (e.g., prices, weather, stock info). If the router classifies a query as `tool`, we route it to a mock function instead of an LLM expert.

In [13]:
import random

# --- Mock Tool Functions ---
def mock_get_crypto_price(coin: str) -> dict:
    """Simulates fetching cryptocurrency price data."""
    prices = {"bitcoin": 67234.50, "ethereum": 3521.80, "solana": 142.35, "dogecoin": 0.162}
    price = prices.get(coin.lower(), round(random.uniform(1, 1000), 2))
    return {"coin": coin.capitalize(), "price_usd": price, "change_24h": round(random.uniform(-5, 5), 2)}


def mock_get_weather(city: str) -> dict:
    """Simulates fetching weather data."""
    conditions = ["Sunny", "Cloudy", "Rainy", "Partly Cloudy", "Windy"]
    return {"city": city, "temperature_f": random.randint(32, 95), "condition": random.choice(conditions)}


TOOL_REGISTRY = {
    "crypto_price": mock_get_crypto_price,
    "weather": mock_get_weather,
}


# --- Updated Router with Tool Category ---
ROUTER_SYSTEM_PROMPT_V2 = (
    "You are an intent classification engine. Your job is to classify user messages "
    "into exactly ONE of these categories: technical, billing, tool, general.\n\n"
    "Rules:\n"
    "- 'technical': Bug reports, code errors, software issues, API problems, debugging.\n"
    "- 'billing': Charges, refunds, subscriptions, invoices, payment issues, pricing.\n"
    "- 'tool': Requests for real-time data like crypto/stock prices, weather, time, etc.\n"
    "- 'general': Greetings, casual chat, general questions, anything else.\n\n"
    "You MUST respond with ONLY a single lowercase word from: technical, billing, tool, general.\n"
    "Do NOT include any explanation, punctuation, or extra text."
)

TOOL_SELECTOR_PROMPT = (
    "You are a tool selection engine. Given a user query that requires fetching real-time data, "
    "determine which tool to call and extract the parameter.\n\n"
    "Available tools:\n"
    "- crypto_price(coin): Get the current price of a cryptocurrency. Parameter: coin name.\n"
    "- weather(city): Get current weather for a city. Parameter: city name.\n\n"
    "Respond in JSON format ONLY: {\"tool\": \"tool_name\", \"parameter\": \"value\"}\n"
    "Example: {\"tool\": \"crypto_price\", \"parameter\": \"bitcoin\"}\n"
    "Do NOT include any other text."
)


def route_prompt_v2(user_input: str) -> str:
    """Enhanced router that includes the 'tool' category."""
    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT_V2},
            {"role": "user", "content": user_input},
        ],
        temperature=0,
        max_tokens=10,
    )
    category = response.choices[0].message.content.strip().lower()
    valid = ["technical", "billing", "tool", "general"]
    if category not in valid:
        print(f"⚠️  Router returned '{category}', falling back to 'general'.")
        category = "general"
    return category


def process_request_v2(user_input: str) -> str:
    """
    Enhanced orchestrator with Tool Use support.
    If the router classifies as 'tool', we select and execute a mock function,
    then pass the tool result to the LLM for a natural-language summary.
    """
    category = route_prompt_v2(user_input)
    print(f"🔀 Router Decision: [{category.upper()}]")

    # Handle tool category
    if category == "tool":
        print("🔧 Activating Tool Use Expert...")
        print("=" * 60)

        # Use LLM to select the right tool and extract the parameter
        tool_response = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[
                {"role": "system", "content": TOOL_SELECTOR_PROMPT},
                {"role": "user", "content": user_input},
            ],
            temperature=0,
            max_tokens=50,
        )
        tool_selection_raw = tool_response.choices[0].message.content.strip()

        try:
            tool_selection = json.loads(tool_selection_raw)
            tool_name = tool_selection["tool"]
            parameter = tool_selection["parameter"]
        except (json.JSONDecodeError, KeyError):
            return f"Sorry, I couldn't determine which tool to use. Raw output: {tool_selection_raw}"

        # Execute the mock tool
        if tool_name in TOOL_REGISTRY:
            tool_result = TOOL_REGISTRY[tool_name](parameter)
            print(f"  📡 Called: {tool_name}(\"{parameter}\")")
            print(f"  📦 Result: {tool_result}")

            # Pass tool result back to LLM for a natural-language response
            summary_response = client.chat.completions.create(
                model=BASE_MODEL,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant. Summarize the following data in a friendly, conversational way."},
                    {"role": "user", "content": f"User asked: \"{user_input}\"\n\nTool returned: {json.dumps(tool_result)}"},
                ],
                temperature=0.7,
                max_tokens=256,
            )
            return summary_response.choices[0].message.content
        else:
            return f"Tool '{tool_name}' is not available."

    # Handle standard expert categories
    expert_config = MODEL_CONFIG[category]
    print(f"🤖 Activating {category.capitalize()} Expert...")
    print("=" * 60)

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert_config["temperature"],
        max_tokens=1024,
    )
    return response.choices[0].message.content


print("✅ Enhanced MoE system with Tool Use ready!")

✅ Enhanced MoE system with Tool Use ready!


In [14]:
# Test Case 4: Tool Use — Crypto Price
print("=" * 60)
print("📩 TEST 4 — Tool Use Query (Crypto)")
print("=" * 60)
query4 = "What is the current price of Bitcoin?"
response4 = process_request_v2(query4)
print(response4)

📩 TEST 4 — Tool Use Query (Crypto)
🔀 Router Decision: [TOOL]
🔧 Activating Tool Use Expert...
  📡 Called: crypto_price("bitcoin")
  📦 Result: {'coin': 'Bitcoin', 'price_usd': 67234.5, 'change_24h': -4.97}
The current price of Bitcoin is around $67,234.50. However, it's worth noting that the price has dropped by about 4.97% over the past 24 hours. Keep an eye on it if you're interested in investing or following the market.


In [15]:
# Test Case 5: Tool Use — Weather
print("=" * 60)
print("📩 TEST 5 — Tool Use Query (Weather)")
print("=" * 60)
query5 = "What's the weather like in New York right now?"
response5 = process_request_v2(query5)
print(response5)

📩 TEST 5 — Tool Use Query (Weather)
🔀 Router Decision: [TOOL]
🔧 Activating Tool Use Expert...
  📡 Called: weather("New York")
  📦 Result: {'city': 'New York', 'temperature_f': 72, 'condition': 'Partly Cloudy'}
I've got the latest update for you about New York. As of now, it's a beautiful day in the Big Apple. The temperature is a pleasant 72 degrees Fahrenheit. And the skies are looking pretty nice too - it's partly cloudy, so you might catch a glimpse of sunshine peeking through the clouds. Sounds like a great day to get out and explore the city!


In [16]:
# Test the v2 pipeline with all categories including tool use
print("\n" + "=" * 60)
print("📩 TEST 6 — Full V2 Pipeline (Technical through enhanced router)")
print("=" * 60)
query6 = "My API keeps returning a 403 Forbidden error."
response6 = process_request_v2(query6)
print(response6)


📩 TEST 6 — Full V2 Pipeline (Technical through enhanced router)
🔀 Router Decision: [TECHNICAL]
🤖 Activating Technical Expert...
**403 Forbidden Error Investigation**

To troubleshoot the issue, let's break down the common causes of a 403 Forbidden error in an API:

1. **Authentication**: Insufficient or incorrect authentication credentials.
2. **Authorization**: Insufficient or incorrect permissions to access the requested resource.
3. **Rate Limiting**: Exceeded rate limits for the API.
4. **IP Blocking**: IP address blocked due to excessive requests or security reasons.

**Step 1: Verify Authentication**

Check if you're using the correct authentication method (e.g., API keys, JWT, OAuth) and providing the correct credentials.

```http
# Example API request with API key authentication
curl -X GET \
  http://example.com/api/data \
  -H 'Authorization: Bearer YOUR_API_KEY'
```

**Step 2: Check Authorization**

Ensure the user or application has the required permissions to access the r